In [2]:
#Install packages

!pip install pretty_midi tensorflow pandas numpy matplotlib

In [3]:
#Imports

import os
import numpy as np
import pandas as pd
import pretty_midi
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [4]:
DATASET_ROOT = "maestro"   # if notebook is in same parent folder
print(DATASET_ROOT)

maestro


In [5]:
#Collect all MIDI file paths

midi_files = []

for root, _, files in os.walk(DATASET_ROOT):
    for file in files:
        if file.lower().endswith(".midi") or file.lower().endswith(".mid"):
            midi_files.append(os.path.join(root, file))

midi_files = sorted(midi_files)

print("Total MIDI files found:", len(midi_files))
print("First 5 files:")
for f in midi_files[:5]:
    print(f)

Total MIDI files found: 1276
First 5 files:
maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi
maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_06_Track06_wav.midi
maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_08_Track08_wav.midi
maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_10_Track10_wav.midi
maestro/2004/MIDI-Unprocessed_SMF_05_R1_2004_01_ORIG_MID--AUDIO_05_R1_2004_02_Track02_wav.midi


In [6]:
# Use only 2011 MIDI files
midi_files = [f for f in midi_files if "/2011/" in f or "\\2011\\" in f]

print("Using all 2011 MIDI files:", len(midi_files))
print("First 10 files:")
for f in midi_files[:10]:
    print(f)

Using all 2011 MIDI files: 163
First 10 files:
maestro/2011/MIDI-Unprocessed_01_R1_2011_MID--AUDIO_R1-D1_02_Track02_wav.midi
maestro/2011/MIDI-Unprocessed_01_R1_2011_MID--AUDIO_R1-D1_03_Track03_wav.midi
maestro/2011/MIDI-Unprocessed_01_R1_2011_MID--AUDIO_R1-D1_04_Track04_wav.midi
maestro/2011/MIDI-Unprocessed_01_R1_2011_MID--AUDIO_R1-D1_05_Track05_wav.midi
maestro/2011/MIDI-Unprocessed_01_R1_2011_MID--AUDIO_R1-D1_06_Track06_wav.midi
maestro/2011/MIDI-Unprocessed_02_R1_2011_MID--AUDIO_R1-D1_08_Track08_wav.midi
maestro/2011/MIDI-Unprocessed_02_R1_2011_MID--AUDIO_R1-D1_09_Track09_wav.midi
maestro/2011/MIDI-Unprocessed_02_R1_2011_MID--AUDIO_R1-D1_10_Track10_wav.midi
maestro/2011/MIDI-Unprocessed_02_R2_2011_MID--AUDIO_R2-D1_02_Track02_wav.midi
maestro/2011/MIDI-Unprocessed_02_R2_2011_MID--AUDIO_R2-D1_03_Track03_wav.midi


In [7]:
#MIDI to notes function

def midi_to_notes(midi_file: str) -> pd.DataFrame:
    pm = pretty_midi.PrettyMIDI(midi_file)

    if len(pm.instruments) == 0:
        return pd.DataFrame(columns=["pitch", "step", "duration"])

    instrument = pm.instruments[0]

    if len(instrument.notes) == 0:
        return pd.DataFrame(columns=["pitch", "step", "duration"])

    notes = []
    sorted_notes = sorted(instrument.notes, key=lambda note: note.start)
    prev_start = sorted_notes[0].start

    for note in sorted_notes:
        start = note.start
        end = note.end

        notes.append({
            "pitch": note.pitch,
            "step": start - prev_start,
            "duration": end - start,
        })

        prev_start = start

    return pd.DataFrame(notes)

In [8]:
#Test one file first

test_notes = midi_to_notes(midi_files[0])
print(test_notes.head())
print("Number of notes:", len(test_notes))

   pitch      step  duration
0     50  0.000000  0.579427
1     38  0.005208  0.621094
2     62  0.263021  0.083333
3     64  0.092448  0.111979
4     66  0.104167  0.134115
Number of notes: 1574


In [9]:
#Sequence function

SEQ_LENGTH = 32

def create_sequences(notes_array, seq_length=32):
    X = []
    y = []

    for i in range(len(notes_array) - seq_length):
        X.append(notes_array[i:i + seq_length])
        y.append(notes_array[i + seq_length])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [10]:
#Compute global normalization stats

global_step_max = 0.0
global_duration_max = 0.0

processed_for_stats = 0
skipped_for_stats = 0

for midi_file in midi_files:
    try:
        notes_df = midi_to_notes(midi_file)

        if notes_df.empty:
            skipped_for_stats += 1
            continue

        global_step_max = max(global_step_max, notes_df["step"].max())
        global_duration_max = max(global_duration_max, notes_df["duration"].max())
        processed_for_stats += 1

    except Exception as e:
        skipped_for_stats += 1
        print(f"Skipping stats for {midi_file}: {e}")

print("Processed for stats:", processed_for_stats)
print("Skipped for stats:", skipped_for_stats)
print("global_step_max =", global_step_max)
print("global_duration_max =", global_duration_max)

Processed for stats: 163
Skipped for stats: 0
global_step_max = 11.802083333333371
global_duration_max = 35.38932291666666


In [11]:
#Build the full training dataset

all_sequences = []
all_targets = []

processed_files = 0
skipped_files = 0

for i, midi_file in enumerate(midi_files):
    try:
        notes_df = midi_to_notes(midi_file)

        if notes_df.empty or len(notes_df) <= SEQ_LENGTH:
            skipped_files += 1
            continue

        notes_array = notes_df[["pitch", "step", "duration"]].values.astype(np.float32)

        # normalize
        notes_array[:, 0] /= 127.0
        notes_array[:, 1] /= global_step_max if global_step_max > 0 else 1.0
        notes_array[:, 2] /= global_duration_max if global_duration_max > 0 else 1.0

        X_file, y_file = create_sequences(notes_array, SEQ_LENGTH)

        if len(X_file) == 0:
            skipped_files += 1
            continue

        all_sequences.append(X_file)
        all_targets.append(y_file)
        processed_files += 1

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(midi_files)} files...")

    except Exception as e:
        skipped_files += 1
        print(f"Skipping {midi_file}: {e}")

X = np.concatenate(all_sequences, axis=0)
y = np.concatenate(all_targets, axis=0)

print("Processed files:", processed_files)
print("Skipped files:", skipped_files)
print("X shape:", X.shape)
print("y shape:", y.shape)

Processed 100/163 files...
Processed files: 163
Skipped files: 0
X shape: (606848, 32, 3)
y shape: (606848, 3)


In [12]:
# Optional train/validation split

print("Total samples:", len(X))

Total samples: 606848


In [13]:
#Positional embedding layer

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, d_model):
        super().__init__()
        self.sequence_length = sequence_length
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length,
            output_dim=d_model
        )

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        pos_embeddings = self.position_embeddings(positions)
        return inputs + pos_embeddings

In [14]:
# Transformer block

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [15]:
#Build the model

def build_transformer_model(seq_length=32, input_dim=3, d_model=64, num_heads=4, ff_dim=128, dropout=0.1):
    inputs = keras.Input(shape=(seq_length, input_dim))

    x = layers.Dense(d_model)(inputs)
    x = PositionalEmbedding(seq_length, d_model)(x)

    x = TransformerBlock(d_model, num_heads, ff_dim, dropout)(x)
    x = TransformerBlock(d_model, num_heads, ff_dim, dropout)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(3)(x)   # predict [pitch, step, duration]

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [16]:
#Create and compile the model

model = build_transformer_model(
    seq_length=SEQ_LENGTH,
    input_dim=3,
    d_model=64,
    num_heads=4,
    ff_dim=128,
    dropout=0.1
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding            │ (None, 32, 64)         │         2,048 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 32, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 32, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,059 (676.01 KB)

 Trainable params: 173,059 (676.01 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
#Train the model

history = model.fit(
    X, y,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    verbose=2
)

Epoch 1/20
7586/7586 - 636s - 84ms/step - loss: 0.0039 - mae: 0.0330 - val_loss: 0.0029 - val_mae: 0.0289
Epoch 2/20
7586/7586 - 612s - 81ms/step - loss: 0.0030 - mae: 0.0290 - val_loss: 0.0028 - val_mae: 0.0286
Epoch 3/20
7586/7586 - 598s - 79ms/step - loss: 0.0030 - mae: 0.0286 - val_loss: 0.0029 - val_mae: 0.0284
Epoch 4/20
7586/7586 - 598s - 79ms/step - loss: 0.0029 - mae: 0.0284 - val_loss: 0.0027 - val_mae: 0.0280
Epoch 5/20
7586/7586 - 708s - 93ms/step - loss: 0.0029 - mae: 0.0283 - val_loss: 0.0028 - val_mae: 0.0275
Epoch 6/20
7586/7586 - 670s - 88ms/step - loss: 0.0029 - mae: 0.0281 - val_loss: 0.0027 - val_mae: 0.0278
Epoch 7/20
7586/7586 - 671s - 88ms/step - loss: 0.0029 - mae: 0.0280 - val_loss: 0.0029 - val_mae: 0.0286
Epoch 8/20
7586/7586 - 645s - 85ms/step - loss: 0.0028 - mae: 0.0279 - val_loss: 0.0027 - val_mae: 0.0271
Epoch 9/20
7586/7586 - 647s - 85ms/step - loss: 0.0028 - mae: 0.0276 - val_loss: 0.0027 - val_mae: 0.0272
Epoch 10/20
7586/7586 - 670s - 88ms/step - los

In [44]:
# Save model
model.save("maestro_transformer_model.keras")
print("Model saved.")

Model saved.


In [4]:
def generate_notes(model, seed_sequence, num_predictions=200,
                   pitch_noise=1.5, step_noise=0.005, duration_noise=0.01):
    generated_notes = []
    current_sequence = seed_sequence.copy()
    last_pitches = []

    allowed_intervals = [-5, -4, -2, 0, 2, 4, 5, 7]

    for _ in range(num_predictions):
        pred = model(current_sequence[np.newaxis, :, :], training=False).numpy()[0]

        pred_pitch = pred[0] * 127.0
        pred_step = pred[1] * global_step_max
        pred_duration = pred[2] * global_duration_max

        pred_pitch += np.random.normal(0, pitch_noise)
        pred_step += np.random.normal(0, step_noise)
        pred_duration += np.random.normal(0, duration_noise)

        pred_pitch = int(np.clip(round(pred_pitch), 48, 84))

        # avoid repeating same note too much, but don't jump randomly
        if len(last_pitches) >= 3 and all(p == pred_pitch for p in last_pitches[-3:]):
            pred_pitch = last_pitches[-1] + np.random.choice(allowed_intervals)
            pred_pitch = int(np.clip(pred_pitch, 48, 84))

        # avoid huge jumps
        if len(last_pitches) > 0:
            previous_pitch = last_pitches[-1]

            # allow bigger movement, but not crazy jumps
            if abs(pred_pitch - previous_pitch) > 14:
                pred_pitch = previous_pitch + np.random.choice([-12, -7, -5, 5, 7, 12])
                pred_pitch = int(np.clip(pred_pitch, 48, 84))

        pred_step = float(np.clip(pred_step, 0.08, 0.25))
        pred_duration = float(np.clip(pred_duration, 0.08, 0.30))

        generated_notes.append({
            "pitch": pred_pitch,
            "step": pred_step,
            "duration": pred_duration
        })

        last_pitches.append(pred_pitch)

        next_note_normalized = np.array([
            pred_pitch / 127.0,
            pred_step / global_step_max if global_step_max > 0 else 0.0,
            pred_duration / global_duration_max if global_duration_max > 0 else 0.0
        ], dtype=np.float32)

        current_sequence = np.vstack([current_sequence[1:], next_note_normalized])

    return generated_notes

In [5]:
def save_generated_notes_to_midi(generated_notes, output_file="generated_from_2013_seed.mid"):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(
        program=pretty_midi.instrument_name_to_program("Acoustic Grand Piano")
    )

    current_time = 0.0

    for note_data in generated_notes:
        current_time += float(note_data["step"])
        start = current_time
        end = start + float(note_data["duration"])

        note = pretty_midi.Note(
            velocity=100,
            pitch=int(note_data["pitch"]),
            start=start,
            end=end
        )

        piano.notes.append(note)

    pm.instruments.append(piano)
    pm.write(output_file)
    print("Saved:", output_file)

In [6]:
# Use first 2013 MIDI file as test seed
all_midi_files = []

for root, _, files in os.walk(DATASET_ROOT):
    for file in files:
        if file.lower().endswith(".midi") or file.lower().endswith(".mid"):
            all_midi_files.append(os.path.join(root, file))

all_midi_files = sorted(all_midi_files)

midi_2013_files = [f for f in all_midi_files if "/2013/" in f or "\\2013\\" in f]

seed_file = midi_2013_files[0]
seed_notes_df = midi_to_notes(seed_file)
seed_array = seed_notes_df[["pitch", "step", "duration"]].values.astype(np.float32)

seed_array[:, 0] /= 127.0
seed_array[:, 1] /= global_step_max if global_step_max > 0 else 1.0
seed_array[:, 2] /= global_duration_max if global_duration_max > 0 else 1.0

seed_sequence = seed_array[:SEQ_LENGTH]

generated_notes = generate_notes(model, seed_sequence, num_predictions=200)
save_generated_notes_to_midi(generated_notes, "generated_from_2013_seed.mid")

print("Seed file:", seed_file)
print("Generated notes:", len(generated_notes))
print(generated_notes[:10])

NameError: name 'os' is not defined